<a href="https://colab.research.google.com/github/Park-gpow/bigdataAn/blob/main/data(04-07_data_save).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 설정 셀 (매 리셋마다 실행)
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json

drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/빅데이터 사례연구")

TRAIN_PATH = Path("/content/drive/MyDrive/빅데이터 사례연구/69.고품질 연구 개발용 이차전지 데이터/Training/TS_LFP(리튬-철-인)/LFP_train_dataset.csv")
VAL_PATH = Path("/content/drive/MyDrive/빅데이터 사례연구/69.고품질 연구 개발용 이차전지 데이터/Validation/TS_LFP(리튬-철-인)/LFP_val_dataset.csv")

TR_OUT_DIR = BASE_DIR / "Tr"
VAL_OUT_DIR = BASE_DIR / "Val"
TS_OUT_DIR = BASE_DIR / "Ts"

TARGET_COL = "discharge_capacity"

HARD_EXCLUDE_COLS = [
    "material_id",
    "DOI",
    "journal_name",
    "chemical_formula",
    "remain_capacity",
    "voltage_range",
]

TARGET_CANDIDATES = [
    "discharge_capacity (mAh/g)",
    "discharge_capacity",
]

print("=" * 60)
print("[설정 셀 점검 완료]")
print("BASE_DIR        :", BASE_DIR)
print("TRAIN_PATH      :", TRAIN_PATH)
print("VAL_PATH        :", VAL_PATH)
print("TRAIN 존재 여부 :", TRAIN_PATH.exists())
print("VAL 존재 여부   :", VAL_PATH.exists())
print("출력 폴더(Tr)   :", TR_OUT_DIR)
print("출력 폴더(Val)  :", VAL_OUT_DIR)
print("출력 폴더(Ts)   :", TS_OUT_DIR)
print("타깃 컬럼 후보  :", TARGET_CANDIDATES)
print("최종 타깃명     :", TARGET_COL)
print("하드 제외 후보  :", HARD_EXCLUDE_COLS)

if TRAIN_PATH.exists() and VAL_PATH.exists():
    print("\n설정 확인 완료: 다음은 원본 로딩 셀 실행")
else:
    print("\n주의: 경로 또는 드라이브 계정을 다시 확인하세요.")
print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[설정 셀 점검 완료]
BASE_DIR        : /content/drive/MyDrive/빅데이터 사례연구
TRAIN_PATH      : /content/drive/MyDrive/빅데이터 사례연구/69.고품질 연구 개발용 이차전지 데이터/Training/TS_LFP(리튬-철-인)/LFP_train_dataset.csv
VAL_PATH        : /content/drive/MyDrive/빅데이터 사례연구/69.고품질 연구 개발용 이차전지 데이터/Validation/TS_LFP(리튬-철-인)/LFP_val_dataset.csv
TRAIN 존재 여부 : True
VAL 존재 여부   : True
출력 폴더(Tr)   : /content/drive/MyDrive/빅데이터 사례연구/Tr
출력 폴더(Val)  : /content/drive/MyDrive/빅데이터 사례연구/Val
출력 폴더(Ts)   : /content/drive/MyDrive/빅데이터 사례연구/Ts
타깃 컬럼 후보  : ['discharge_capacity (mAh/g)', 'discharge_capacity']
최종 타깃명     : discharge_capacity
하드 제외 후보  : ['material_id', 'DOI', 'journal_name', 'chemical_formula', 'remain_capacity', 'voltage_range']

설정 확인 완료: 다음은 원본 로딩 셀 실행


In [6]:
# 원본 로딩 확인
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(VAL_PATH)

print("train_raw shape:", train_raw.shape)
print("test_raw shape :", test_raw.shape)
print("원본 LFP_val_dataset.csv 는 최종 테스트셋(test_raw)으로 사용합니다.")

display(train_raw.head())
display(test_raw.head())

train_raw shape: (131790, 48)
test_raw shape : (16474, 48)
원본 LFP_val_dataset.csv 는 최종 테스트셋(test_raw)으로 사용합니다.


,material_id,material_structure,synthesis_method,sintering_T1(C),sintering_t1(h),Li_source,Co_source,Mn_source,Ni_source,electrolyte,...,perc_barrier_2d,perc_radius_1d,perc_radius_2d,max_packing_eff,chemical_ordering,struct_hetero_bond,struct_hetero_cell,remain_capacity,voltage_range(V)_min,voltage_range(V)_max
0,sb-qm02g6seq327,Olivine,Other,80.000,12.000,Other,not included,not included,not included,PEGDA:LiTFSI:FEC (30wt%:30wt%:40wt%),...,3.551874,0.420391,0.283390,0.244128,0.625636,0.088872,0.141162,32.7375,2.5,4.0
1,sb-s0p64tj6loib,Olivine,Other,481.439,8.975,Other,not included,not included,not included,EC:DEC (1v:1v)/1M LiPF6,...,3.489572,0.433367,0.324282,0.252252,0.676321,0.077316,0.122146,7.3750,2.5,4.0
2,sb-yqj0n0jkkxxu,Olivine,Other,800.000,6.000,LiOH·H2O,not included,not included,not included,EC:DMC:DEC (1v:1v:1v)/1M LiPF6,...,3.595708,0.490158,0.268195,0.206849,0.466781,0.122647,0.202444,120.0000,2.5,4.2
3,sb-5btluxs1d42z,Olivine,Sol-gel,850.000,0.500,Other,not included,not included,not included,LP40,...,3.601212,0.459863,0.281722,0.227974,0.557776,0.103705,0.168746,58.5000,3.5,2.0
4,sb-3fuphm5i9fut,Olivine,Other,800.000,6.000,LiOH·H2O,not included,not included,not included,EC:DMC:DEC (1v:1v:1v)/1M LiPF6,...,3.708188,0.531832,0.284856,0.202029,0.452119,0.127125,0.213342,100.0800,2.5,4.2


,material_id,material_structure,synthesis_method,sintering_T1(C),sintering_t1(h),Li_source,Co_source,Mn_source,Ni_source,electrolyte,...,perc_barrier_2d,perc_radius_1d,perc_radius_2d,max_packing_eff,chemical_ordering,struct_hetero_bond,struct_hetero_cell,remain_capacity,voltage_range(V)_min,voltage_range(V)_max
0,sb-jk58m5n04yj6,Olivine,Other,80.000,12.000,Other,not included,not included,not included,EC:DMC (1v:1v)/1M LiPF6,...,3.623038,0.472537,0.283500,0.224029,0.542631,0.107125,0.175519,80.750,2.00,3.8
1,sb-xu0ekkelzyqn,Olivine,Other,80.000,12.000,Other,not included,not included,not included,PEGDA:LiTFSI:FEC (30wt%:30wt%:40wt%),...,3.614280,0.490430,0.326120,0.233384,0.602137,0.093967,0.154952,46.230,2.50,4.0
2,sb-xi4uwvn96oyb,Olivine,Other,481.439,8.975,Other,not included,not included,not included,EC:DEC (1v:1v)/1M LiPF6,...,3.580580,0.463155,0.313250,0.238561,0.617161,0.090655,0.147343,33.275,2.50,4.0
3,sb-em8dg75wbqax,Other,Other,481.439,8.975,Other,not included,not included,not included,LiTFSI-PEO-0.2AIF3,...,3.518177,0.424067,0.306702,0.250227,0.662102,0.080647,0.127620,12.680,2.00,3.8
4,sb-1iu1iuowuqzv,Olivine,Other,50.000,5.000,Other,not included,not included,not included,EC:EMC (3w:7w)/1M LiPF6,...,3.692962,0.537833,0.278308,0.195008,0.421663,0.133281,0.224031,135.000,2.95,3.8


In [7]:
# split first: train_raw만 80:20 분할
from sklearn.model_selection import train_test_split

train_dev_raw, val_dev_raw = train_test_split(
    train_raw,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("train_dev_raw shape:", train_dev_raw.shape)
print("val_dev_raw shape  :", val_dev_raw.shape)
print("test_raw shape     :", test_raw.shape)
print("random_state=42 기준으로 train_raw 80:20 분할 완료")

train_dev_raw shape: (105432, 48)
val_dev_raw shape  : (26358, 48)
test_raw shape     : (16474, 48)
random_state=42 기준으로 train_raw 80:20 분할 완료


In [8]:
# =========================================================
# 이미 분할된 raw 데이터를 Google Drive에 저장만 하는 셀
# - split 없음
# - 저장만 수행
# =========================================================

# 전제:
# train_dev_raw, val_dev_raw, test_raw 가 이미 만들어져 있어야 함

TR_OUT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
TS_OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_RAW_SAVE_PATH = TR_OUT_DIR / "train_dev_raw.csv"
VAL_RAW_SAVE_PATH = VAL_OUT_DIR / "val_dev_raw.csv"
TEST_RAW_SAVE_PATH = TS_OUT_DIR / "test_raw.csv"

train_dev_raw.to_csv(TRAIN_RAW_SAVE_PATH, index=False, encoding="utf-8-sig")
val_dev_raw.to_csv(VAL_RAW_SAVE_PATH, index=False, encoding="utf-8-sig")
test_raw.to_csv(TEST_RAW_SAVE_PATH, index=False, encoding="utf-8-sig")

print("=" * 70)
print("[raw 분할본 저장 완료]")
print("train_dev_raw shape :", train_dev_raw.shape)
print("val_dev_raw shape   :", val_dev_raw.shape)
print("test_raw shape      :", test_raw.shape)
print()
print("저장 위치")
print(" -", TRAIN_RAW_SAVE_PATH)
print(" -", VAL_RAW_SAVE_PATH)
print(" -", TEST_RAW_SAVE_PATH)
print("=" * 70)

[raw 분할본 저장 완료]
train_dev_raw shape : (105432, 48)
val_dev_raw shape   : (26358, 48)
test_raw shape      : (16474, 48)

저장 위치
 - /content/drive/MyDrive/빅데이터 사례연구/Tr/train_dev_raw.csv
 - /content/drive/MyDrive/빅데이터 사례연구/Val/val_dev_raw.csv
 - /content/drive/MyDrive/빅데이터 사례연구/Ts/test_raw.csv
